In [72]:
import pandas as pd

In [73]:
def load_stock(path):
    # Load the CSV file
    df = pd.read_csv(path)  # replace with your actual file name

    # Handle missing values: drop rows with any missing data
    df.dropna(inplace=True)

    # Sort data by supplier
    df = df.sort_values(by='supplier')
    
    return df

In [74]:
class Drug:
    def __init__(self, filepath):
        # Store the dataframe as an instance attribute
        self.drugs = pd.read_csv(filepath)
        # Convert expiry_date to datetime for proper comparisons
        self.drugs['expiry_date'] = pd.to_datetime(self.drugs['expiry_date'])
        # You can also call a summary method here if needed
        # self.summary = self.drug_summary()   # if you define drug_summary
        self.total_drugs = len(self.drugs)
        
    def price_summary(self):
        """Return most expensive and cheapest drug details."""
        df = self.drugs
        most_expensive = df.loc[df['unit_price_usd'].idxmax()]
        cheapest = df.loc[df['unit_price_usd'].idxmin()]
        return {
            "most_expensive": most_expensive.to_dict(),  # convert Series to dict
            "cheapest": cheapest.to_dict()
        }

    def supplier_summary(self):
        df = self.drugs
        supplier_dict = {}
        for supplier, group in df.groupby('supplier'):
            supplier_dict[supplier] = {
                'list_of_drugs': group['drug_name'].tolist(),
                'total_number': int(group['quantity_in_stock'].sum())
            }
        counts = {
            'form': df['form'].value_counts().to_dict(),
            'category': df['category'].value_counts().to_dict(),
            'supplier': df['supplier'].value_counts().to_dict()
        }
        return {'supplier_details': supplier_dict, 'counts': counts}
    
    def total_inventory_value(self, by_supplier=False):
        """
        Calculate the total inventory value (quantity * unit_price).
        
        Args:
            by_supplier (bool): If True, return a dictionary with values per supplier.
        
        Returns:
            float or dict: Total value (float) if by_supplier=False,
                        otherwise dict {supplier: total_value}.
        """
        df = self.drugs
        df['total_value'] = df['quantity_in_stock'] * df['unit_price_usd']
        
        if by_supplier:
            return df.groupby('supplier')['total_value'].sum().to_dict()
        else:
            return df['total_value'].sum()
        
    def description(self):
        """Return a string describing the Drug class and its data."""
        return (
            f"Drug Class: Manages pharmaceutical drug data.\n"
            f"Loaded {self.total_drugs} drug records.\n"
            f"Available methods: price_summary(), supplier_summary().\n"
            f"Use this class to analyze drug prices and supplier information."
        )


In [75]:
class Inventory:
    
    def __init__(self, df):
        # Store the dataframe as an instance attribute so all methods can access it
        self.df = df.copy()  # copy to avoid modifying original
        # Ensure expiry_date is datetime for consistent comparisons
        self.df['expiry_date'] = pd.to_datetime(self.df['expiry_date'])
        # Store total items as an attribute (optional)
        self.total_items = self.df['quantity_in_stock'].sum()
        self.total_drugs = len(self.df)
    
    
    
    def get_low_stock(self):
        
        df = self.df
        return df[(df['quantity_in_stock'] <= df['reorder_level']) & 
                  (df['reorder_level'] - df['quantity_in_stock'] <= 15)]
    
    def get_re_stock(self):
        
        df = self.df
        return df[df['quantity_in_stock'] <= df['reorder_level'] + 20]
    
    def stock_summary(self):
        """
        Return a dictionary with details of most/least stock and newest/oldest expiry.
        """
        df = self.df
        most_stock = df.loc[df['quantity_in_stock'].idxmax()].to_dict()
        least_stock = df.loc[df['quantity_in_stock'].idxmin()].to_dict()
        latest_expiry = df.loc[df['expiry_date'].idxmax()].to_dict()
        oldest_expiry = df.loc[df['expiry_date'].idxmin()].to_dict()

        return {
            "most_stock": most_stock,
            "least_stock": least_stock,
            "latest_expiry": latest_expiry,
            "oldest_expiry": oldest_expiry
        }
    
    def check_expiry(self):
        
        today = pd.Timestamp.now().normalize()
        expired = self.df[self.df["expiry_date"] < today]
        return expired

    def near_expiry(self, days=30):
        
        today = pd.Timestamp.now().normalize()
        threshold = today + pd.Timedelta(days=days)
        near_exp = self.df[(self.df['expiry_date'] > today) & 
                        (self.df['expiry_date'] <= threshold)]
        return near_exp

    def expiry_loss_forecast(self, days=90):
        
        expiring = self.near_expiry(days)
        
        if expiring.empty:
            return 0.0
        # Compute total value (quantity * unit price)
        total_value = (expiring['quantity_in_stock'] * expiring['unit_price_usd']).sum()
        return total_value
    
    def description(self):
        return (
            f"Inventory Class: Manages drug inventory operations.\n"
            f"Currently tracking {self.total_drugs} drug types with {self.total_items} total items in stock.\n"
            f"Available methods: check_expiry(), get_low_stock(), get_re_stock(), stock_summary(), near_expiry(), description()."
        )
    
    def restock_report(self, filename="restock_report.txt"):
        
        lines = []
        lines.append(f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
        lines.append("")

        # Basic counts (from instance attributes)
        lines.append(f"Total drug types: {self.total_drugs}")
        lines.append(f"Total items in stock: {self.total_items}")
        lines.append("")

        # Total inventory value (compute directly)
        total_value = (self.df['quantity_in_stock'] * self.df['unit_price_usd']).sum()
        lines.append(f"Total inventory value: ${total_value:,.2f}")
        lines.append("")

        # Expired drugs (using check_expiry)
        expired = self.check_expiry()
        lines.append(f"Expired drugs: {len(expired)}")
        if not expired.empty:
            lines.append("  " + ", ".join(expired['drug_name'].tolist()))
        lines.append("")

        # Low stock (within 15 below reorder) using get_low_stock
        low_stock = self.get_low_stock()
        lines.append(f"Low stock drugs (within 15 below reorder): {len(low_stock)}")
        if not low_stock.empty:
            for _, row in low_stock.iterrows():
                lines.append(f"  {row['drug_name']}: {row['quantity_in_stock']} (reorder at {row['reorder_level']})")
        lines.append("")

        # Restock soon (≤ reorder + 20) using get_re_stock
        restock = self.get_re_stock()
        lines.append(f"Drugs needing restock soon (≤ reorder+20): {len(restock)}")
        if not restock.empty:
            for _, row in restock.iterrows():
                lines.append(f"  {row['drug_name']}: {row['quantity_in_stock']} (reorder at {row['reorder_level']})")
        lines.append("")

        # Expiry loss forecasts using expiry_loss_forecast
        lines.append("Expiry loss forecast (value of drugs expiring within...):")
        for days in [30, 60, 90]:
            loss = self.expiry_loss_forecast(days)
            lines.append(f"  Next {days} days: ${loss:,.2f}")
        lines.append("")

        # Top 5 most stocked drugs
        top_stock = self.df.nlargest(5, 'quantity_in_stock')[['drug_name', 'quantity_in_stock']]
        lines.append("Top 5 most stocked drugs:")
        for _, row in top_stock.iterrows():
            lines.append(f"  {row['drug_name']}: {row['quantity_in_stock']} units")
        lines.append("")

        # Top 5 most valuable drugs (by current stock value)
        # Add temporary column without altering original df
        df_temp = self.df.copy()
        df_temp['stock_value'] = df_temp['quantity_in_stock'] * df_temp['unit_price_usd']
        top_value = df_temp.nlargest(5, 'stock_value')[['drug_name', 'stock_value']]
        lines.append("Top 5 most valuable drugs (by current stock value):")
        for _, row in top_value.iterrows():
            lines.append(f"  {row['drug_name']}: ${row['stock_value']:,.2f}")
        lines.append("")

        lines.append("=" * 60)

        # Write to file with UTF‑8 encoding
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))

        # Print to console
        # print(f"\nRestock report saved to {filename}")
        return "\n".join(lines)
        

    def expiry_report(self, days=90, filename="expiry_report.txt"):
        
        lines = []
        lines.append(f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
        lines.append(f"Forecast window: {days} days")
        lines.append("")

        # ---------- Expired drugs ----------
        expired = self.check_expiry()
        lines.append("EXPIRED DRUGS")
        lines.append("-" * 40)
        if expired.empty:
            lines.append("  No expired drugs.")
        else:
            lines.append(f"  Total expired items: {len(expired)}")
            lines.append("  List:")
            for _, row in expired.iterrows():
                lines.append(f"    {row['drug_name']} (expired on {row['expiry_date'].date()}) – {row['quantity_in_stock']} units")
        lines.append("")

        # ---------- Near‑expiry drugs (within `days`) ----------
        near = self.near_expiry(days)
        lines.append(f"DRUGS EXPIRING WITHIN {days} DAYS")
        lines.append("-" * 40)
        if near.empty:
            lines.append(f"  No drugs expire within the next {days} days.")
        else:
            lines.append(f"  Total items: {len(near)}")
            lines.append("  List by expiry date:")
            # Sort by expiry date for clarity
            near_sorted = near.sort_values('expiry_date')
            for _, row in near_sorted.iterrows():
                lines.append(f"    {row['drug_name']} – expires {row['expiry_date'].date()} – {row['quantity_in_stock']} units")
        lines.append("")

        # ---------- Loss forecast ----------
        loss = self.expiry_loss_forecast(days)
        lines.append("LOSS FORECAST")
        lines.append("-" * 40)
        lines.append(f"  Total value of drugs expiring within {days} days: ${loss:,.2f}")
        lines.append("")

        # ---------- Breakdown by category (optional) ----------
        if not near.empty:
            # Add temporary value column
            near_with_value = near.copy()
            near_with_value['value'] = near_with_value['quantity_in_stock'] * near_with_value['unit_price_usd']

            # By category
            cat_sum = near_with_value.groupby('category')['value'].sum().sort_values(ascending=False)
            lines.append("EXPIRING VALUE BY CATEGORY")
            lines.append("-" * 40)
            for category, val in cat_sum.items():
                lines.append(f"  {category}: ${val:,.2f}")
            lines.append("")

            # Top 5 most valuable expiring drugs
            top_value = near_with_value.nlargest(5, 'value')[['drug_name', 'value']]
            lines.append("TOP 5 MOST VALUABLE EXPIRING DRUGS")
            lines.append("-" * 40)
            for _, row in top_value.iterrows():
                lines.append(f"  {row['drug_name']}: ${row['value']:,.2f}")
            lines.append("")

        # Write to file with UTF‑8 encoding
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))

        #DRUG INVENTORY RESTOCK REPORT
       # print(f"\nExpiry report saved to {filename}")
        return "\n".join(lines)
        

In [76]:

def generate_restock_report(Inventory):

    return Inventory.restock_report()


In [77]:
def generate_expiry_report(Inventory):

    return Inventory.expiry_report()
